In [1]:
from src.dataset import DigitAnomalyDataset
from src.encoders import PCAAngleEncoder, AmplitudeEncoder
from src.feature_map import AngleFeatureMap, AmplitudeFeatureMap
from src.kernel import FidelityQuantumKernel, ClassicalRBFKernel
from src.models import OneClassSVMAnomalyModel
from src.anomaly_eval import AnomalyEvaluator
from src.visualizer import AnomalyVisualizer
from src.experiment import QuantumAnomalyExperiment
import pandas as pd

In [4]:
def run_all_four_methods(
    dataset,
    pca_components=4,
    reps=2,
    nu=0.1,
    threshold_strategy="expected_fraction",
    verbose=False,
    plot_best=False,
):
    """
    Run all 4 combinations:

        1. PCA encoder       + quantum angle kernel
        2. Amplitude encoder + quantum fidelity kernel
        3. PCA encoder       + classical RBF kernel
        4. Amplitude encoder + classical RBF kernel

    Returns
    -------
    comparison : pandas.DataFrame
        Summary table of metrics.

    results : dict
        Full result dictionaries for each method.

    experiments : dict
        Experiment objects, useful for plotting/debugging.
    """

    methods = {
        "PCA + Quantum Angle Kernel": {
            "encoder": PCAAngleEncoder(n_components=pca_components),
            "kernel": FidelityQuantumKernel(
                feature_map=AngleFeatureMap(reps=reps)
            ),
        },

        "Amplitude + Quantum Fidelity Kernel": {
            "encoder": AmplitudeEncoder(n_qubits=6),
            "kernel": FidelityQuantumKernel(
                feature_map=AmplitudeFeatureMap()
            ),
        },

        "PCA + Classical RBF Kernel": {
            "encoder": PCAAngleEncoder(n_components=pca_components),
            "kernel": ClassicalRBFKernel(gamma="scale"),
        },

        "Amplitude + Classical RBF Kernel": {
            "encoder": AmplitudeEncoder(n_qubits=6),
            "kernel": ClassicalRBFKernel(gamma="scale"),
        },
    }

    results = {}
    experiments = {}
    rows = []

    for method_name, components in methods.items():
        exp = QuantumAnomalyExperiment(
            name=method_name,
            dataset=dataset,
            encoder=components["encoder"],
            kernel=components["kernel"],
            model=OneClassSVMAnomalyModel(nu=nu),
            evaluator=AnomalyEvaluator(threshold_strategy=threshold_strategy),
            visualizer=AnomalyVisualizer(),
        )

        result = exp.run(verbose=verbose)

        results[method_name] = result
        experiments[method_name] = exp

        metrics = result["metrics"]
        encoder_info = result.get("encoder_info", {})
        kernel_info = result.get("kernel_info", {})

        row = {
            "method": method_name,
            "encoder": encoder_info.get("type", "unknown"),
            "kernel": kernel_info.get("type", "unknown"),
            "auc": metrics["auc"],
            "f1": metrics["f1"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "threshold": metrics["threshold"],
        }

        if encoder_info.get("type") == "pca_angle":
            row["encoded_dim"] = encoder_info.get("n_components")
            row["qubits_or_features"] = encoder_info.get("n_components")
            row["explained_variance"] = encoder_info.get("total_explained_variance")

        elif encoder_info.get("type") == "amplitude":
            row["encoded_dim"] = encoder_info.get("state_dim")
            row["qubits_or_features"] = encoder_info.get("n_qubits")
            row["explained_variance"] = None

        else:
            row["encoded_dim"] = None
            row["qubits_or_features"] = None
            row["explained_variance"] = None

        rows.append(row)

    comparison = pd.DataFrame(rows)
    comparison = comparison.sort_values("auc", ascending=False).reset_index(drop=True)

    print("\n===== Comparison Table =====")
    display(comparison)

    if plot_best:
        best_method = comparison.iloc[0]["method"]
        print(f"\nPlotting best method: {best_method}")
        experiments[best_method].plot_results()

    return comparison, results, experiments

In [5]:
dataset = DigitAnomalyDataset(
    normal_digit=0,
    anomaly_digits=[1, 2, 3, 4, 5, 6, 7, 8, 9],
    n_train_normal=120,
    n_test_normal=40,
    n_test_anomaly=120,
    random_state=7,
)

comparison, results, experiments = run_all_four_methods(
    dataset=dataset,
    pca_components=4,
    reps=2,
    nu=0.1,
    verbose=False,
    plot_best=False,
)


===== Comparison Table =====


/Users/asaad/Documents/Algolabs/IQcodefest_Stagiaires/src/kernel.py:335: ComplexWarning: Casting complex values to real discards the imaginary part
  X_train = np.asarray(X_train, dtype=float)
/Users/asaad/Documents/Algolabs/IQcodefest_Stagiaires/src/kernel.py:346: ComplexWarning: Casting complex values to real discards the imaginary part
  X_test = np.asarray(X_test, dtype=float)


,method,encoder,kernel,auc,f1,precision,recall,threshold,encoded_dim,qubits_or_features,explained_variance
0,Amplitude + Classical RBF Kernel,amplitude,classical_rbf_kernel,0.997708,0.991667,0.991667,0.991667,0.951009,64,6,NaN
1,Amplitude + Quantum Fidelity Kernel,amplitude,fidelity_quantum_kernel,0.997500,0.991667,0.991667,0.991667,0.764504,64,6,NaN
2,PCA + Classical RBF Kernel,pca_angle,classical_rbf_kernel,0.984792,0.966667,0.966667,0.966667,0.102438,4,4,0.491227
3,PCA + Quantum Angle Kernel,pca_angle,fidelity_quantum_kernel,0.942917,0.916667,0.916667,0.916667,0.031216,4,4,0.491227


In [6]:
dataset = DigitAnomalyDataset(
    normal_digit=0,
    anomaly_digits=[6, 9],
    n_train_normal=120,
    n_test_normal=40,
    n_test_anomaly=120,
    random_state=7,
)

comparison, results, experiments = run_all_four_methods(
    dataset=dataset,
    pca_components=4,
    reps=2,
    nu=0.1,
    verbose=False,
    plot_best=False,
)


===== Comparison Table =====


/Users/asaad/Documents/Algolabs/IQcodefest_Stagiaires/src/kernel.py:335: ComplexWarning: Casting complex values to real discards the imaginary part
  X_train = np.asarray(X_train, dtype=float)
/Users/asaad/Documents/Algolabs/IQcodefest_Stagiaires/src/kernel.py:346: ComplexWarning: Casting complex values to real discards the imaginary part
  X_test = np.asarray(X_test, dtype=float)


,method,encoder,kernel,auc,f1,precision,recall,threshold,encoded_dim,qubits_or_features,explained_variance
0,Amplitude + Classical RBF Kernel,amplitude,classical_rbf_kernel,0.991875,0.991667,0.991667,0.991667,0.779101,64,6,NaN
1,Amplitude + Quantum Fidelity Kernel,amplitude,fidelity_quantum_kernel,0.988958,0.983333,0.983333,0.983333,0.623943,64,6,NaN
2,PCA + Classical RBF Kernel,pca_angle,classical_rbf_kernel,0.954792,0.933333,0.933333,0.933333,-0.014849,4,4,0.491227
3,PCA + Quantum Angle Kernel,pca_angle,fidelity_quantum_kernel,0.952500,0.933333,0.933333,0.933333,0.056891,4,4,0.491227


In [7]:
dataset = DigitAnomalyDataset(
    normal_digit=1,
    anomaly_digits=[7],
    n_train_normal=120,
    n_test_normal=40,
    n_test_anomaly=120,
    random_state=7,
)

comparison, results, experiments = run_all_four_methods(
    dataset=dataset,
    pca_components=4,
    reps=2,
    nu=0.1,
    verbose=False,
    plot_best=False,
)


===== Comparison Table =====


/Users/asaad/Documents/Algolabs/IQcodefest_Stagiaires/src/kernel.py:335: ComplexWarning: Casting complex values to real discards the imaginary part
  X_train = np.asarray(X_train, dtype=float)
/Users/asaad/Documents/Algolabs/IQcodefest_Stagiaires/src/kernel.py:346: ComplexWarning: Casting complex values to real discards the imaginary part
  X_test = np.asarray(X_test, dtype=float)


,method,encoder,kernel,auc,f1,precision,recall,threshold,encoded_dim,qubits_or_features,explained_variance
0,Amplitude + Classical RBF Kernel,amplitude,classical_rbf_kernel,0.999375,0.991667,0.991667,0.991667,0.510416,64,6,NaN
1,Amplitude + Quantum Fidelity Kernel,amplitude,fidelity_quantum_kernel,0.995625,0.991667,0.991667,0.991667,0.258749,64,6,NaN
2,PCA + Quantum Angle Kernel,pca_angle,fidelity_quantum_kernel,0.855625,0.866667,0.866667,0.866667,-0.048931,4,4,0.5993
3,PCA + Classical RBF Kernel,pca_angle,classical_rbf_kernel,0.498333,0.691667,0.691667,0.691667,-0.429914,4,4,0.5993


In [8]:
dataset = DigitAnomalyDataset(
    normal_digit=3,
    anomaly_digits=[8],
    n_train_normal=120,
    n_test_normal=40,
    n_test_anomaly=120,
    random_state=7,
)

comparison, results, experiments = run_all_four_methods(
    dataset=dataset,
    pca_components=4,
    reps=2,
    nu=0.1,
    verbose=False,
    plot_best=False,
)


===== Comparison Table =====


/Users/asaad/Documents/Algolabs/IQcodefest_Stagiaires/src/kernel.py:335: ComplexWarning: Casting complex values to real discards the imaginary part
  X_train = np.asarray(X_train, dtype=float)
/Users/asaad/Documents/Algolabs/IQcodefest_Stagiaires/src/kernel.py:346: ComplexWarning: Casting complex values to real discards the imaginary part
  X_test = np.asarray(X_test, dtype=float)


,method,encoder,kernel,auc,f1,precision,recall,threshold,encoded_dim,qubits_or_features,explained_variance
0,PCA + Quantum Angle Kernel,pca_angle,fidelity_quantum_kernel,0.948750,0.908333,0.908333,0.908333,-0.007707,4,4,0.472628
1,Amplitude + Classical RBF Kernel,amplitude,classical_rbf_kernel,0.896042,0.908333,0.908333,0.908333,-0.293046,64,6,NaN
2,Amplitude + Quantum Fidelity Kernel,amplitude,fidelity_quantum_kernel,0.874792,0.883333,0.883333,0.883333,-0.390000,64,6,NaN
3,PCA + Classical RBF Kernel,pca_angle,classical_rbf_kernel,0.743542,0.800000,0.800000,0.800000,-0.356999,4,4,0.472628
